In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, roc_auc_score, accuracy_score,
    ConfusionMatrixDisplay, RocCurveDisplay,
)


In [ ]:
# ======================
# 1. ПАРАМЕТРЫ
# ======================
DATA = Path("../data/credit_risk.csv")
FIG = Path("../figures")
FIG.mkdir(parents=True, exist_ok=True)
RANDOM_SEED = 11
TEST_SIZE = 0.25


In [ ]:
data = pd.read_csv(DATA)


In [ ]:
print("Number of rows:", data.shape[0])
print("Columns:", list(data.columns))


In [ ]:
data.info()
data.head()


In [ ]:
data.describe().round(2)


In [ ]:
print("default rate:", round(data["default"].mean(), 3))
print(data["default"].value_counts())


In [ ]:
# Круговая диаграмма дисбаланса
counts = data["default"].value_counts().sort_index()
colors = sns.color_palette("rocket", 2)

plt.figure(figsize=(7, 6))
ax = plt.subplot(111)
wedges, _ = ax.pie(
    counts,
    startangle=90,
    colors=colors,
    wedgeprops=dict(width=0.45, edgecolor="w"),
)
legend_labels = [
    f"{'Нет дефолта' if i == 0 else 'Дефолт'} ({c} — {c / counts.sum() * 100:.1f}%)"
    for i, c in counts.items()
]
plt.legend(wedges, legend_labels, loc="center left", bbox_to_anchor=(1, 0.5))
plt.title("Дисбаланс классов (default)", fontsize=14, pad=15)
centre = plt.Circle((0, 0), 0.25, color="white")
ax.add_artist(centre)
plt.tight_layout()
plt.savefig(FIG / "credit_default_pie.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# late_payments vs default
plt.figure(figsize=(8, 5))
sns.boxplot(data=data, x="default", y="late_payments", palette="rocket", hue="default", legend=False)
plt.title("Просрочки (late_payments) по классу default", fontsize=14, pad=15)
plt.xlabel("default", fontsize=12)
plt.ylabel("late_payments", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(FIG / "credit_late_payments_box.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ======================
# 2. ПРИЗНАКИ И SPLIT
# ======================
feature_cols = ["age", "income", "debt_ratio", "num_cards", "late_payments", "employment_years"]
X = data[feature_cols]
y = data["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_SEED
)
print("train default rate:", round(y_train.mean(), 3))
print("test  default rate:", round(y_test.mean(), 3))


In [ ]:
# ======================
# 3. МОДЕЛЬ БЕЗ ВЕСОВ
# ======================
pipe_none = Pipeline([
    ("sc", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, class_weight=None)),
])
pipe_none.fit(X_train, y_train)
pred_none = pipe_none.predict(X_test)
proba_none = pipe_none.predict_proba(X_test)[:, 1]

print("no_weight accuracy", round(accuracy_score(y_test, pred_none), 3))
print("no_weight AUC", round(roc_auc_score(y_test, proba_none), 3))
print(classification_report(y_test, pred_none, digits=3))


In [ ]:
# ======================
# 4. МОДЕЛЬ С class_weight='balanced'
# ======================
pipe_bal = Pipeline([
    ("sc", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced")),
])
pipe_bal.fit(X_train, y_train)
pred_bal = pipe_bal.predict(X_test)
proba_bal = pipe_bal.predict_proba(X_test)[:, 1]

print("balanced accuracy", round(accuracy_score(y_test, pred_bal), 3))
print("balanced AUC", round(roc_auc_score(y_test, proba_bal), 3))
print(classification_report(y_test, pred_bal, digits=3))


In [ ]:
# ======================
# 5. ВИЗУАЛИЗАЦИЯ СРАВНЕНИЯ
# ======================
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ConfusionMatrixDisplay.from_predictions(y_test, pred_none, ax=axes[0], colorbar=False)
axes[0].set_title("Без весов")

ConfusionMatrixDisplay.from_predictions(y_test, pred_bal, ax=axes[1], colorbar=False)
axes[1].set_title("class_weight=balanced")

plt.tight_layout()
plt.savefig(FIG / "credit_confusion_compare.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
RocCurveDisplay.from_predictions(y_test, proba_none, name="no_weight", ax=ax)
RocCurveDisplay.from_predictions(y_test, proba_bal, name="balanced", ax=ax)
ax.set_title("ROC: без весов vs balanced", fontsize=13)
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(FIG / "credit_roc_compare.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
print("Accuracy на дисбалансе обманчива — смотрим precision/recall/AUC.")
print("class_weight применяем после простого baseline.")
